In [ ]:
# Import necessary packages
import os
import re
import sys
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pathlib import Path

## Define file paths

In [ ]:
notebook_dir = Path.cwd()
functions_dir = notebook_dir.parent / "Functions"
if str(functions_dir) not in sys.path:
    sys.path.append(str(functions_dir))
from coregister import standardize_and_align, stack_rgb_and_dsm, run_local_coregistration, run_global_coregistration, calculate_intersection_and_master, resample_to_master_grid

In [ ]:
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
original_image_path = os.path.join(base_path, "0_Original_Images")

flights = {}

# 1. Search all folders
for folder_name in os.listdir(original_image_path):
    folder_path = os.path.join(original_image_path, folder_name)
    
    if os.path.isdir(folder_path) and "Burned_Orfanos" in folder_name:
        
        match = re.search(r'\d{6}', folder_name)
        if not match:
            continue
            
        date_str = match.group()
        
        dsm_path = None
        rgb_path = None
        
        for file in os.listdir(folder_path):
            if file.lower().endswith('.tif'):
                if "dsm.tif" in file.lower():
                    dsm_path = os.path.join(folder_path, file)
                if "true_ortho.tif" in file.lower() or "mosaic.tif" in file.lower():  
                    rgb_path = os.path.join(folder_path, file)
                
                    
        if dsm_path and rgb_path:
            flights[date_str] = {
                "rgb": rgb_path,
                "dsm": dsm_path
            }

for date, paths in flights.items():
    print(f"Date: {date}")
    print(f"  -> RGB: {paths['rgb']}")
    print(f"  -> DSM: {paths['dsm']}\n")

## Aligning No-Data Values

In [ ]:
stacked_dir = os.path.join(base_path, "1_Stacked_Raw")
os.makedirs(stacked_dir, exist_ok=True)

print("--- STEP 1: ALIGNING NODATA VALUES TO -0.0 ---")

for date, paths in flights.items():
    print(f"Aligning Flight {date}...")
    
    # Clean both files and update the dictionary with the new paths
    flights[date]["rgb"] = standardize_and_align(paths["rgb"])
    flights[date]["dsm"] = standardize_and_align(paths["dsm"])

print("✅ All files aligned to float32 and -0.0!\n")

## Check if alignment was successfull

In [ ]:
for date in flights.keys():

    with rasterio.open(flights[date]["dsm"]) as dsm:
        print(f"{date} DSM No-Data: {dsm.nodata}")

    with rasterio.open(flights[date]["rgb"]) as rgb:
        print(f"{date} RGB No-Data: {rgb.nodata}")

# Step 1: Stack RGB and DSM and align CRS

In [ ]:
stacked_dir = os.path.join(base_path, "1_Stacked_Raw")
os.makedirs(stacked_dir, exist_ok=True)

master_rgb = flights["250821"]["rgb"]
with rasterio.open(master_rgb) as master_src:
    target_crs = master_src.crs

for date, paths in flights.items():

    rgb_in = paths["rgb"]
    dsm_in = paths["dsm"]
    stacked_out = os.path.join(stacked_dir, f"{date}_stacked.tif")

    print(f"\nStacking Flight {date}...")

    stack_rgb_and_dsm(rgb_in, dsm_in, target_crs, stacked_out, nodatavalue=0.0)

In [ ]:
for date, paths in flights.items():
    with rasterio.open(paths['dsm']) as src:
        dsm_arr = src.read(1)
        
        # 1. Read metadata
        nodata_tag = src.nodata
        
        # 2. Check for NaNs
        has_nans = np.isnan(dsm_arr).any()
        nan_count = np.count_nonzero(np.isnan(dsm_arr))
        
        # 3. Count NoData values
        nodata_count = 0
        if nodata_tag is not None:
            nodata_count = np.count_nonzero(dsm_arr == nodata_tag)
            
        # 4. Filter valid pixels for accurate statistics
        # We create a boolean mask that ignores both NaNs and NoData values
        valid_mask = ~np.isnan(dsm_arr)
        if nodata_tag is not None:
            valid_mask &= (dsm_arr != nodata_tag)
            
        # Extract only the valid pixels into a 1D array
        valid_pixels = dsm_arr[valid_mask]
        
        # 5. Calculate statistics safely
        if valid_pixels.size > 0:
            min_val = np.min(valid_pixels)
            max_val = np.max(valid_pixels)
            mean_val = np.mean(valid_pixels)
            std_val = np.std(valid_pixels)
        else:
            min_val = max_val = mean_val = std_val = "N/A (Empty Raster)"
        
        # 6. Print the analysis
        print(f"--- Image Date: {date} ---")
        print(f"  Official NoData Tag:       {nodata_tag}")
        print(f"  NoData Pixel Count:        {nodata_count}")
        print(f"  Contains NaNs?             {has_nans} (Count: {nan_count})")
        print("  --- Valid Pixel Metrics ---")
        
        if valid_pixels.size > 0:
            print(f"  Minimum Value:             {min_val:.2f}")
            print(f"  Maximum Value:             {max_val:.2f}")
            print(f"  Mean Value:                {mean_val:.2f}")
            print(f"  Standard Deviation:        {std_val:.2f}\n")
        else:
            print("  No valid pixels available to calculate metrics.\n")

# Step 2: Create Master Reference Grid

In [ ]:
# 1. Define target resolution (2.5cm)
TARGET_GSD = 0.025

# 2. Calculate Master Grid
files = os.listdir(stacked_dir)
master_transform, master_width, master_height, master_crs = calculate_intersection_and_master(stacked_dir, files, TARGET_GSD)

print(f"Final Grid: {master_width}x{master_height} Pixel.")

# 3. Create Output Folder
out_dir = os.path.join(base_path, "2_Master_Reference")
os.makedirs(out_dir, exist_ok=True)

# 4. Resample october image
october_filename = "251026_stacked.tif" 
img_path = os.path.join(stacked_dir, october_filename)
master_reference = os.path.join(out_dir, "251026_master_reference.tif")

resample_to_master_grid(
    img_path, 
    master_reference, 
    master_transform, 
    master_width, 
    master_height, 
    master_crs
)

# STEP 2: Apply Co-Registraion

## Step 2.1: Global first 

In [ ]:
kwargs_global = {
    'r_b4match': 1,
    's_b4match': 1,
    'max_shift': 200,
    'ws': (2048, 2048),
    'wp': (557132.29, 4219339.59),  # heres the road intersection
    'fmt_out': 'GTIFF',
    'nodata': (0, 0),
    'match_gsd': False,
    'align_grids': False,
    'resamp_alg_calc': 'cubic',
    'resamp_alg_deshift': 'cubic',
    'q': False,
}

In [ ]:
# 1. Define Paths
global_dir = os.path.join(base_path, "3_Coregistrated/1_Global")

# 2. Define target paths
target_files = [
    os.path.join(stacked_dir, file) 
    for file in os.listdir(stacked_dir) 
    if not file.startswith("251026") and file.endswith("_stacked.tif")
]

# 3. Run global Coregistration
run_global_coregistration(master_reference, target_files, global_dir, kwargs_global)

## Step 2.2: Local Co-Registration

In [ ]:
kwargs_local = {
    'grid_res': 100,
    'window_size': (512, 512),
    'min_reliability': 35,
    'max_shift': 50,
    'fmt_out': 'GTIFF',
    'q': False,
    'match_gsd': False,
    'resamp_alg_calc': 'cubic',
    'resamp_alg_deshift': 'cubic'
}

In [ ]:
# 1. Paths
local_dir = os.path.join(base_path, "3_Coregistrated/2_Local")

# 2. Get target image paths
target_files = []
for folder in os.listdir(global_dir):
    folder_path = os.path.join(global_dir, folder)
    if os.path.isdir(folder_path):
        global_tif = os.path.join(folder_path, f"{folder}_coreg.tif")
        if os.path.exists(global_tif):
            target_files.append(global_tif)
            
# 4. Run local coregistration
run_local_coregistration(master_reference, target_files, local_dir, kwargs_local)

# Step 3: Resample all images to same resolution

In [ ]:
# --- 1. DEFINE PATHS ---
master_dir = os.path.join(base_path, "2_Master_Reference")
master_reference = os.path.join(master_dir, "251026_master_reference.tif")
local_dir = os.path.join(base_path, "3_Coregistrated/2_Local")
out_dir = os.path.join(base_path, "4_Resampled_Grids")
os.makedirs(out_dir, exist_ok=True)
TARGET_GSD = 0.025
NODATA_VAL = -9999.0
out_dtype = "float32"

# Collect all files + reference raster
all_files = glob.glob(os.path.join(local_dir, "*", "*_local_coreg.tif"))
all_files.append(master_reference)

print(f"Start final Grid-Snapping for {len(all_files)} images...")

# --- 2. CALCULATING THE INTERSECTION ---
lefts, bottoms, rights, tops = [], [], [], []
master_crs = None

for file in all_files:
    with rasterio.open(file) as src:
        lefts.append(src.bounds.left)
        bottoms.append(src.bounds.bottom)
        rights.append(src.bounds.right)
        tops.append(src.bounds.top)
        if master_crs is None:
            master_crs = src.crs

common_left = max(lefts)
common_bottom = max(bottoms)
common_right = min(rights)
common_top = min(tops)

# --- 3. CALCULATING NEW PERFECT GRID ---
master_transform = from_origin(common_left, common_top, TARGET_GSD, TARGET_GSD)
master_width = int((common_right - common_left) / TARGET_GSD)
master_height = int((common_top - common_bottom) / TARGET_GSD)

print(f"Final ML-Matrix: {master_height} Rows x {master_width} Columns (Resolution: {TARGET_GSD}m)")
print("=========================================\n")


# --- 4. PRESS IMAGES IN NEW GRID ---
for file in all_files:
    
    basename = os.path.basename(file).split("_")[0]
    out_path = os.path.join(out_dir, f"{basename}_resampled.tif")
    
    print(f"Snapping Flight {basename} in Master-Grid...")
    
    with rasterio.open(file) as src:
        profile = src.profile.copy()
        profile.update({
            "crs": master_crs,
            "transform": master_transform,
            "width": master_width,
            "height": master_height,
            "nodata": NODATA_VAL,
            "dtype": out_dtype,
            "compress": "lzw"
        })

        # Empty array for 4 channels  (RGB + DSM)
        data = np.full((src.count, master_height, master_width), 0.0, dtype=out_dtype)

        # Resampling all bands
        reproject(
            source=rasterio.band(src, tuple(range(1, src.count + 1))),
            destination=data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=master_transform,
            dst_crs=master_crs,
            src_nodata=0.0,
            dst_nodata=0.0,
            resampling=Resampling.cubic
        )

        # Set no data value to -9999.0
        data[data == 0.0] = NODATA_VAL

        # Clip RGB to [0,255]
        for i in range(3):  
            valid_mask = data[i] != NODATA_VAL
            # Only clip valid pixels
            data[i][valid_mask] = np.clip(data[i][valid_mask], 0, 255)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


print("✅ All grids successfully snapped and ML-ready!")

# SANITY CHECK

In [ ]:
# --- PATHS ---
folder_path = os.path.join(base_path, "4_Resampled_Grids") 
all_files = sorted(glob.glob(os.path.join(folder_path, "*_resampled.tif")))

print("==================================================")
print("🚀 ULTIMATE SANITY CHECK")
print("==================================================\n")

# Master reference variables
ref_shape = None
ref_res = None
ref_crs = None
TARGET_NODATA = -9999.0 

for idx, file in enumerate(all_files):
    basename = os.path.basename(file)
    print(f"--- Checking: {basename} ---")
    
    with rasterio.open(file) as src:
        if idx == 0:
            ref_shape = (src.height, src.width)
            ref_res = src.res
            ref_crs = src.crs
            print( "  [MASTER REFERENCE LOADED]")
            print(f"  Dimensions: {ref_shape[1]}x{ref_shape[0]}, GSD: {ref_res[0]}m")
            print(f"  CRS: {ref_crs}\n")
        
        # 1. GEOMETRY CHECK
        shape_match = (src.height, src.width) == ref_shape
        res_match = src.res == ref_res
        crs_match = src.crs == ref_crs
        
        print( "  Geometry Check:")
        print(f"    {'✅' if shape_match else '❌'} Dimensions (Expected: {ref_shape})")
        print(f"    {'✅' if res_match else '❌'} Pixel Resolution (Expected: {ref_res})")
        print(f"    {'✅' if crs_match else '❌'} Coordinate System")
        
        # 2. BAND & NODATA CHECK
        bands_match = src.count == 4
        nodata_match = src.nodata == TARGET_NODATA
        
        print( "  Metadata Check:")
        print(f"    {'✅' if bands_match else '❌'} 4 Channels present")
        print(f"    {'✅' if nodata_match else '❌'} NoData Tag is {TARGET_NODATA}")
        
        # 3. DATA HEALTH CHECK (Filtering out NoData first!)
        rgb = src.read((1, 2, 3))
        dsm = src.read(4)
        
        # Extract only valid pixels to calculate real min/max
        valid_rgb = rgb[rgb != TARGET_NODATA]
        valid_dsm = dsm[dsm != TARGET_NODATA]

        has_nans = np.isnan(valid_rgb).any() or np.isnan(valid_dsm).any()
        
        # RGB Check (Should be within 0-255)
        rgb_min = np.min(valid_rgb) if valid_rgb.size > 0 else 0
        rgb_max = np.max(valid_rgb) if valid_rgb.size > 0 else 0
        rgb_ok = -0.1 <= rgb_min and rgb_max <= 256.0
        
        # DSM Check (Reasonable earth elevation values)
        dsm_min = np.min(valid_dsm) if valid_dsm.size > 0 else 0
        dsm_max = np.max(valid_dsm) if valid_dsm.size > 0 else 0
        dsm_ok = dsm_min > -100 and dsm_max < 9000 
        
        print( "  Data Health Check:")
        print(f"    {'✅' if not has_nans else '❌'} No NaNs present")
        print(f"    {'✅' if rgb_ok else '❌'} RGB Range ({rgb_min:.1f} to {rgb_max:.1f})")
        print(f"    {'✅' if dsm_ok else '❌'} DSM Range ({dsm_min:.2f}m to {dsm_max:.2f}m)")
        
        # FINAL VERDICT
        if all([shape_match, res_match, crs_match, bands_match, nodata_match, rgb_ok, dsm_ok]):
            print("  -> STATUS: READY FOR MACHINE LEARNING 🟢\n")
        else:
            print("  -> STATUS: ERRORS DETECTED 🔴\n")